# tensor-zeros-init — ex8: z-buffer painter with per-step debug

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tensor-zeros-init`. Running the final beacon cell reports progress against the `Numpy: Core array literacy` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Core array literacy` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`tensor-zeros-init`** (exercise 8). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-zeros-init"
DD_SUBTOPIC = "Numpy: Core array literacy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## torch zero-init — quick refresher

**The allocate-then-scatter pattern.** Pre-allocate a buffer of the right `(shape, dtype, device)` with `t.zeros(...)`, then write per-element results into it via indexed assignment or `index_add_` / `scatter_add_`. This is faster and clearer than `list.append` + `t.stack`, and it's the canonical move for histograms, confusion matrices, depth buffers, and any per-ray accumulator.

**Dtype matters.** Default is `float32`. Index buffers MUST be `t.long`. Counters should be `t.long` (or `t.int64`). Use `t.zeros_like(x)` when you want a fresh buffer that mirrors `x.shape + x.dtype + x.device` exactly.

### Exercise 8 — z-buffer painter with per-step debug

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Combine `t.zeros` + `t.zeros_like` + indexed conditional write to build a per-pixel depth buffer; return the buffer history for debug.
> Keywords: z-buffer, depth-test, ray-tracing, multi-step-debug
> ```

**KCs targeted:** `zeros-1d-shape`, `zeros-like-mirrors-input`, `zeros-allocate-then-fill`

Implement `ex8_zbuffer(num_pixels, objects)`. A miniature Ray Tracing z-buffer that you can step through:

1. Allocate a `(num_pixels,)` float zero buffer named `z_buf`, then fill it with `+inf` (no object yet → infinitely far).
2. Allocate a `(num_pixels,)` `t.long` zero buffer named `obj_id` (which object owns each pixel; 0 means 'none').
3. For each `(name, pixel_idxs, depths)` in `objects`, perform a depth test: where `depths < z_buf[pixel_idxs]`, overwrite both `z_buf[pixel_idxs]` and `obj_id[pixel_idxs]`.
4. Record a `(z_buf.clone(), obj_id.clone())` snapshot after each object so the caller can replay the painter.

Inputs:
- `num_pixels`: int.
- `objects`: list of `(name: str, pixel_idxs: long Tensor (K,), depths: float Tensor (K,))`. `name` is for printing only; objects are numbered 1, 2, ... in input order.

Output: a dict with keys `'z_buf'` (final), `'obj_id'` (final), and `'history'` (list of `(name, z_buf_clone, obj_id_clone)` after each step). Also print the per-step pixel-ownership count so the caller sees the painter evolve.

> ⚠️ **Integrative exercise.** Combines 3 KCs (shape, zeros_like, indexed conditional write) plus a debug-introspection loop. Expect a step up vs Exercises 1-5.

In [ ]:
def ex8_zbuffer(num_pixels: int, objects: list) -> dict:
    z_buf = t.zeros(num_pixels)
    z_buf.fill_(float('inf'))
    obj_id = t.zeros(num_pixels, dtype=t.long)
    history = []
    for i, (name, pixel_idxs, depths) in enumerate(objects, start=1):
        closer = depths < z_buf[pixel_idxs]
        winning_pixels = pixel_idxs[closer]
        winning_depths = depths[closer]
        z_buf[winning_pixels] = winning_depths
        obj_id[winning_pixels] = i
        owned = (obj_id != 0).sum().item()
        print(f'  step {i} ({name}): {owned}/{num_pixels} pixels owned')
        history.append((name, z_buf.clone(), obj_id.clone()))
    return {'z_buf': z_buf, 'obj_id': obj_id, 'history': history}


<details><summary>Solution</summary>

```python
def ex8_zbuffer(num_pixels: int, objects: list) -> dict:
    z_buf = t.zeros(num_pixels)
    z_buf.fill_(float('inf'))
    obj_id = t.zeros(num_pixels, dtype=t.long)
    history = []
    for i, (name, pixel_idxs, depths) in enumerate(objects, start=1):
        closer = depths < z_buf[pixel_idxs]
        winning_pixels = pixel_idxs[closer]
        winning_depths = depths[closer]
        z_buf[winning_pixels] = winning_depths
        obj_id[winning_pixels] = i
        owned = (obj_id != 0).sum().item()
        print(f'  step {i} ({name}): {owned}/{num_pixels} pixels owned')
        history.append((name, z_buf.clone(), obj_id.clone()))
    return {'z_buf': z_buf, 'obj_id': obj_id, 'history': history}
```

**The depth-test pattern.** This is the integer-arithmetic heart of a Ray Tracing renderer: every closest-hit query is a per-pixel depth test against a sentinel-initialised buffer. Sentinel = `+inf` (so any real hit wins). Owner-id starts at 0 (none).

**Why clone the history.** `z_buf` and `obj_id` are mutated in place by subsequent steps. If you store the live references instead of clones, every snapshot will end up identical to the final state — a classic alias bug. The test's mutation-after-the-fact assertion catches it.

**Why the per-step print.** Multi-step pipelines fail silently in the middle. Logging `owned / num_pixels` after each object turns the loop into a self-narrating debug trace — essential when you later replace static `objects` with a real scene.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex8'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex8',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()